In [5]:
import time
from enum import Enum
from typing import Optional, Callable, TypedDict, Literal
from langgraph.graph import StateGraph, START, END


# ==========================================
# 1. Circuit Breaker 共享类定义
# ==========================================
class CircuitState(Enum):
    CLOSED = "CLOSED"
    OPEN = "OPEN"
    HALF_OPEN = "HALF_OPEN"


class FakeClock:
    def __init__(self, start_time: float = 1000.0):
        self.now = start_time

    def time(self) -> float:
        return self.now

    def advance(self, seconds: float):
        self.now += seconds


class CircuitBreaker:
    def __init__(
        self,
        failure_threshold: int = 3,
        cooldown: float = 30.0,
        clock_fn: Callable[[], float] = time.time,
    ):
        self.failure_threshold = failure_threshold
        self.cooldown = cooldown
        self.clock_fn = clock_fn

        self.state = CircuitState.CLOSED
        self.failure_count = 0
        self.opened_at: Optional[float] = None
        self.half_open_probe_in_flight = False

    def can_call(self) -> bool:
        now = self.clock_fn()

        if self.state == CircuitState.CLOSED:
            return True

        if self.state == CircuitState.OPEN:
            if self.opened_at is None:
                raise RuntimeError(
                    "[CircuitBreaker Error] Invalid state: OPEN but 'opened_at' is None."
                )

            elapsed = now - self.opened_at
            if elapsed < self.cooldown:
                return False

            print(
                f"[CircuitBreaker] ⏱️ Cooldown 结束 ({elapsed:.2f}s >= {self.cooldown}s)，状态转换: OPEN -> HALF_OPEN"
            )
            self.state = CircuitState.HALF_OPEN
            self.half_open_probe_in_flight = True
            return True

        if self.state == CircuitState.HALF_OPEN:
            if self.half_open_probe_in_flight:
                print(
                    "[CircuitBreaker] 🛡️ HALF_OPEN 状态已有试探请求在执行中，拦截并发请求 (Fast Fail)"
                )
                return False

            self.half_open_probe_in_flight = True
            return True

        return False

    def record_success(self):
        if self.state == CircuitState.HALF_OPEN:
            print(
                "[CircuitBreaker] 🟢 HALF_OPEN 试探成功！服务恢复，状态转换: HALF_OPEN -> CLOSED"
            )
        else:
            print("[CircuitBreaker] 🟢 请求成功，重置失败计数")

        self.state = CircuitState.CLOSED
        self.failure_count = 0
        self.opened_at = None
        self.half_open_probe_in_flight = False

    def record_failure(self):
        now = self.clock_fn()

        if self.state == CircuitState.HALF_OPEN:
            print(
                "[CircuitBreaker] 🚨 HALF_OPEN 试探失败！重新触发熔断，状态转换: HALF_OPEN -> OPEN"
            )
            self.state = CircuitState.OPEN
            self.opened_at = now
            self.half_open_probe_in_flight = False
            return

        if self.state == CircuitState.CLOSED:
            self.failure_count += 1
            print(
                f"[CircuitBreaker] ⚠️ 连续失败计次: {self.failure_count}/{self.failure_threshold}"
            )
            if self.failure_count >= self.failure_threshold:
                print(
                    f"[CircuitBreaker] 🚨 连续失败达阈值 ({self.failure_threshold})！状态转换: CLOSED -> OPEN"
                )
                self.state = CircuitState.OPEN
                self.opened_at = now
                self.half_open_probe_in_flight = False
            return

        if self.state == CircuitState.OPEN:
            print("[CircuitBreaker] ℹ️ 处于 OPEN 状态，忽略无效的 failure 记录")
            return


# 全局共享实例与 Mock 句柄
fake_clock = FakeClock(start_time=1000.0)
hr_api_breaker = CircuitBreaker(
    failure_threshold=3, cooldown=30.0, clock_fn=fake_clock.time
)
mock_api_should_succeed = True
mock_api_called = False


# ==========================================
# 2. Agent State 与 LangGraph 节点构建
# ==========================================
class WorkflowState(TypedDict):
    employee_id: str
    amount: float
    gate_action: Optional[Literal["ALLOW", "FAST_FAIL"]]
    api_status: Optional[int]
    result: Optional[str]


def check_circuit_breaker_node(state: WorkflowState):
    if hr_api_breaker.can_call():
        print("[Gate Node] ✅ Circuit Breaker 放行 (ALLOW)")
        return {"gate_action": "ALLOW"}
    else:
        print("[Gate Node] ⛔ Circuit Breaker 拦截 (FAST_FAIL)")
        return {"gate_action": "FAST_FAIL"}


def fast_fail_fallback_node(state: WorkflowState):
    print("[Fallback Node] 🚀 Fast Fail 触发，立即返回降级结果 (不等待 API 超时)")
    return {"result": "FALLBACK: Downstream HR Service is temporarily unavailable (Circuit Open)"}


def call_hr_api_node(state: WorkflowState):
    global mock_api_called
    mock_api_called = True

    if mock_api_should_succeed:
        print(f"[HR API] 200 OK - 成功发放薪资 {state['amount']} 至 Employee {state['employee_id']}")
        return {"api_status": 200}
    else:
        print(f"[HR API] 503 Service Unavailable - Downstream failure")
        return {"api_status": 503}


def record_success_node(state: WorkflowState):
    hr_api_breaker.record_success()
    return {"result": "SUCCESS: Salary transferred successfully."}


def record_failure_node(state: WorkflowState):
    hr_api_breaker.record_failure()
    return {"result": "FALLBACK: HR API execution failed, routing to manual review."}


# 路由逻辑
def route_gate(state: WorkflowState) -> Literal["call_hr_api", "fast_fail_fallback_node"]:
    if state["gate_action"] == "ALLOW":
        return "call_hr_api"
    return "fast_fail_fallback_node"


def route_api_result(state: WorkflowState) -> Literal["record_success_node", "record_failure_node"]:
    if state["api_status"] == 200:
        return "record_success_node"
    return "record_failure_node"


# 构图
builder = StateGraph(WorkflowState)
builder.add_node("check_circuit_breaker", check_circuit_breaker_node)
builder.add_node("fast_fail_fallback_node", fast_fail_fallback_node)
builder.add_node("call_hr_api", call_hr_api_node)
builder.add_node("record_success_node", record_success_node)
builder.add_node("record_failure_node", record_failure_node)

builder.add_edge(START, "check_circuit_breaker")
builder.add_conditional_edges(
    "check_circuit_breaker",
    route_gate,
    {
        "call_hr_api": "call_hr_api",
        "fast_fail_fallback_node": "fast_fail_fallback_node",
    },
)
builder.add_conditional_edges(
    "call_hr_api",
    route_api_result,
    {
        "record_success_node": "record_success_node",
        "record_failure_node": "record_failure_node",
    },
)
builder.add_edge("fast_fail_fallback_node", END)
builder.add_edge("record_success_node", END)
builder.add_edge("record_failure_node", END)

graph = builder.compile()


# ==========================================
# 3. 核心调用：5 个 Scenario 的 graph.invoke() 验证
# ==========================================
if __name__ == "__main__":
    print("=" * 60)
    print("Scenario 1: Breaker = CLOSED, API success -> CLOSED")
    print("=" * 60)
    mock_api_should_succeed = True
    mock_api_called = False

    res1 = graph.invoke({"employee_id": "EMP-001", "amount": 5000.0})
    print(f"---> 状态验证: Breaker State = {hr_api_breaker.state.value} | HR API Called = {mock_api_called}")
    print(f"---> 执行结果: {res1['result']}\n")


    print("=" * 60)
    print("Scenario 2: 连续 3 次失败触发熔断 -> CLOSED -> OPEN")
    print("=" * 60)
    mock_api_should_succeed = False

    for i in range(1, 4):
        print(f"--- Req {i} ---")
        graph.invoke({"employee_id": f"EMP-00{i+1}", "amount": 1000.0})
    print(f"---> 状态验证: Breaker State = {hr_api_breaker.state.value}\n")


    print("=" * 60)
    print("Scenario 3: OPEN 状态下新请求到达 -> FAST_FAIL (验证 API 零调用)")
    print("=" * 60)
    mock_api_called = False

    res3 = graph.invoke({"employee_id": "EMP-099", "amount": 2000.0})
    print(f"---> 状态验证: Breaker State = {hr_api_breaker.state.value} | HR API Called = {mock_api_called}")
    print(f"---> 执行结果: {res3['result']}\n")


    print("=" * 60)
    print("Scenario 4: Cooldown 到期(瞬间推进31s) -> HALF_OPEN -> Probe Success -> CLOSED")
    print("=" * 60)
    # 使用 FakeClock 瞬间跳过 31 秒
    fake_clock.advance(31.0)
    mock_api_should_succeed = True
    mock_api_called = False

    res4 = graph.invoke({"employee_id": "EMP-005", "amount": 5000.0})
    print(f"---> 状态验证: Breaker State = {hr_api_breaker.state.value} | HR API Called = {mock_api_called}")
    print(f"---> 执行结果: {res4['result']}\n")


    print("=" * 60)
    print("Scenario 5: Cooldown 到期 -> HALF_OPEN -> Probe 503 -> OPEN (重新计时)")
    print("=" * 60)
    # 再次制造 3 次失败触发 OPEN
    mock_api_should_succeed = False
    for _ in range(3):
        graph.invoke({"employee_id": "EMP-MOCK", "amount": 100.0})
    
    # 瞬间推进 31s
    fake_clock.advance(31.0)
    
    # Probe 试探请求再次失败
    graph.invoke({"employee_id": "EMP-PROBE", "amount": 100.0})
    print(f"---> 状态验证: Breaker State = {hr_api_breaker.state.value}")

Scenario 1: Breaker = CLOSED, API success -> CLOSED
[Gate Node] ✅ Circuit Breaker 放行 (ALLOW)
[HR API] 200 OK - 成功发放薪资 5000.0 至 Employee EMP-001
[CircuitBreaker] 🟢 请求成功，重置失败计数
---> 状态验证: Breaker State = CLOSED | HR API Called = True
---> 执行结果: SUCCESS: Salary transferred successfully.

Scenario 2: 连续 3 次失败触发熔断 -> CLOSED -> OPEN
--- Req 1 ---
[Gate Node] ✅ Circuit Breaker 放行 (ALLOW)
[HR API] 503 Service Unavailable - Downstream failure
[CircuitBreaker] ⚠️ 连续失败计次: 1/3
--- Req 2 ---
[Gate Node] ✅ Circuit Breaker 放行 (ALLOW)
[HR API] 503 Service Unavailable - Downstream failure
[CircuitBreaker] ⚠️ 连续失败计次: 2/3
--- Req 3 ---
[Gate Node] ✅ Circuit Breaker 放行 (ALLOW)
[HR API] 503 Service Unavailable - Downstream failure
[CircuitBreaker] ⚠️ 连续失败计次: 3/3
[CircuitBreaker] 🚨 连续失败达阈值 (3)！状态转换: CLOSED -> OPEN
---> 状态验证: Breaker State = OPEN

Scenario 3: OPEN 状态下新请求到达 -> FAST_FAIL (验证 API 零调用)
[Gate Node] ⛔ Circuit Breaker 拦截 (FAST_FAIL)
[Fallback Node] 🚀 Fast Fail 触发，立即返回降级结果 (不等待 API 超时)
---> 状态验证: Br